<a href="https://colab.research.google.com/github/Spoorti803/Deep_Reinforcement_Learning/blob/main/PPO_LunarLander_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install -q "gymnasium[box2d]" stable-baselines3

1. **gymnasium**: provides the LunarLander environment.
2. **stable-baselines3**: provides the PPO algorithm and neural-network policy implementation.

In [17]:
import gymnasium as gym
env = gym.make('LunarLander-v3')

observation, info = env.reset()

print("Initial observation: ",observation)
print("Observation shape: ",observation.shape)
print("Action space:",env.action_space)


Initial observation:  [ 3.4399034e-04  1.4221020e+00  3.4839828e-02  4.9696723e-01
 -3.9193770e-04 -7.8917434e-03  0.0000000e+00  0.0000000e+00]
Observation shape:  (8,)
Action space: Discrete(4)


You should see an observation shape of (8,) and an action space similar to Discrete(4).<br>
The four actions are:<br>
1. Do nothing
2. Fire left orientation engine
3. Fire main engine
4. Fire right orientation engine

In [18]:
#before training the PPO let us check what happens when randomly actions are taken
for step in range(20):
  action = env.action_space.sample()

  observation, reward, terminated, truncated, info = env.step(action)
  print("Step: ",step)
  print("Action: ",action)
  print("Reward: ",reward)

  if terminated or truncated:
    observation, info = env.reset()

env.close()


Step:  0
Action:  0
Reward:  1.4663849319956057
Step:  1
Action:  3
Reward:  1.3705916794952497
Step:  2
Action:  3
Reward:  1.1833228754247511
Step:  3
Action:  2
Reward:  -2.475952800170961
Step:  4
Action:  0
Reward:  1.3606663338072167
Step:  5
Action:  1
Reward:  1.657649671380825
Step:  6
Action:  2
Reward:  -3.72775467312606
Step:  7
Action:  3
Reward:  1.2841289985667583
Step:  8
Action:  3
Reward:  1.1899162811631083
Step:  9
Action:  0
Reward:  1.3913802291880586
Step:  10
Action:  3
Reward:  0.8906314849818397
Step:  11
Action:  3
Reward:  0.5686782962180541
Step:  12
Action:  2
Reward:  -4.931876960552915
Step:  13
Action:  3
Reward:  0.5219627979653307
Step:  14
Action:  3
Reward:  0.11830358282986822
Step:  15
Action:  2
Reward:  -3.8852587048460974
Step:  16
Action:  0
Reward:  0.5307536183600234
Step:  17
Action:  2
Reward:  -2.9531883773912453
Step:  18
Action:  2
Reward:  -3.0324759750511303
Step:  19
Action:  3
Reward:  -0.3941833191151989


`env.step(action)` applies the choosen action and returns 5 things:
1. observation: the new state.
2. reward: the feedback received.
3. terminated: whether the episode ended naturally, for example through landing or crashing.
4. truncated: whether the episode stopped because of a limit, such as the time limit.
5. info: additional diagnostic information.

### Train the Deep RL

In [20]:
#create the PPO model

from stable_baselines3 import PPO

env = gym.make('LunarLander-v3')



## Result-1

In [25]:
model20k = PPO("MlpPolicy",env,verbose=1)
model20k.learn(total_timesteps=20_000)
model20k.save("ppo_Lunarlander-res1")
print("training finished and model saved")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96.3     |
|    ep_rew_mean     | -222     |
| time/              |          |
|    fps             | 587      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 94.8         |
|    ep_rew_mean          | -200         |
| time/                   |              |
|    fps                  | 528          |
|    iterations           | 2            |
|    time_elapsed         | 7            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0051592346 |
|    clip_fraction        | 0.0167       |
|    clip_range           | 0.2          |
|    e

In [26]:

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display
import os

video_folder = "videos_20k"

eval_env = gym.make(
    "LunarLander-v3",
    render_mode="rgb_array"
)

eval_env = RecordVideo(
    eval_env,
    video_folder=video_folder,
    episode_trigger=lambda episode_id: True
)

observation, info = eval_env.reset()
terminated = False
truncated = False
total_reward = 0

while not (terminated or truncated):
    action, _ = model20k.predict(
        observation,
        deterministic=True
    )

    observation, reward, terminated, truncated, info = (
        eval_env.step(action)
    )

    total_reward += reward

eval_env.close()

print("Total episode reward:", round(total_reward, 2))

video_files = [
    f for f in os.listdir(video_folder)
    if f.endswith(".mp4")
]

if video_files:
    video_path = os.path.join(video_folder, video_files[-1])
    display(Video(video_path, embed=True))


Total episode reward: -1984.47


## Result-2



In [29]:
model200k = PPO("MlpPolicy",env,verbose=1)
model200k.learn(total_timesteps=200_000)
model200k.save("ppo_Lunarlander-re2")
print("training finished and model saved")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96.1     |
|    ep_rew_mean     | -208     |
| time/              |          |
|    fps             | 696      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 95.3         |
|    ep_rew_mean          | -199         |
| time/                   |              |
|    fps                  | 500          |
|    iterations           | 2            |
|    time_elapsed         | 8            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0059889117 |
|    clip_fraction        | 0.0152       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | -0.00446     |
|    learning_r

In [30]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display
import os

video_folder = "videos200k"

eval_env = gym.make(
    "LunarLander-v3",
    render_mode="rgb_array"
)

eval_env = RecordVideo(
    eval_env,
    video_folder=video_folder,
    episode_trigger=lambda episode_id: True
)

observation, info = eval_env.reset()
terminated = False
truncated = False
total_reward = 0

while not (terminated or truncated):
    action, _ = model200k.predict(
        observation,
        deterministic=True
    )

    observation, reward, terminated, truncated, info = (
        eval_env.step(action)
    )

    total_reward += reward

eval_env.close()

print("Total episode reward:", round(total_reward, 2))

video_files = [
    f for f in os.listdir(video_folder)
    if f.endswith(".mp4")
]

if video_files:
    video_path = os.path.join(video_folder, video_files[-1])
    display(Video(video_path, embed=True))


Total episode reward: 161.25
